# Настройка DuckLake

In [ ]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [ ]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

In [ ]:
%sql duckdb:///:memory:

# Создание подключения к DuckLake

In [ ]:
%%sql
INSTALL ducklake;

In [ ]:
%%sql
ATTACH 'ducklake:my_ducklake.ducklake' AS my_ducklake;
USE my_ducklake;

# Создание таблицы в DuckLake

In [ ]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

In [ ]:
%%sql
FROM fake_data

# Спецификация DuckLake

[Specification/Tables](https://ducklake.select/docs/stable/specification/tables/overview)

![](https://ducklake.select/images/schema/ducklake-schema-v1.0-light.svg)

In [ ]:
%%sql
SELECT
    table_catalog,
    table_name
FROM information_schema.tables

In [ ]:
%%sql
SELECT current_catalog()

In [ ]:
%%sql
USE '__ducklake_metadata_my_ducklake'

In [ ]:
%%sql
FROM ducklake_snapshot_changes

In [ ]:
%%sql
FROM ducklake_snapshot

In [ ]:
%%sql
FROM ducklake_data_file

In [ ]:
df = %sql FROM ducklake_data_file WHERE end_snapshot IS NULL

In [ ]:
pd.read_parquet(f'my_ducklake.ducklake.files/main/fake_data/{df.path[0]}')

# Уборка в DuckLake
- [Expire Snapshots](https://ducklake.select/docs/stable/duckdb/maintenance/expire_snapshots)
- [Cleanup of Files](https://ducklake.select/docs/stable/duckdb/maintenance/cleanup_of_files)

In [ ]:
%%sql
FROM ducklake_snapshot

In [ ]:
%%sql
CALL ducklake_expire_snapshots('my_ducklake', older_than => now() - INTERVAL '1 minute');

In [ ]:
%%sql
FROM ducklake_snapshot

In [ ]:
%%sql
CALL ducklake_cleanup_old_files(
    'my_ducklake',
    cleanup_all => true
);

In [ ]:
%%sql
SELECT current_catalog()

# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [ ]:
%%sql
USE 'my_ducklake';

In [ ]:
%%sql
FROM fake_data

In [ ]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

In [ ]:
%%sql
FROM fake_data

In [ ]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

In [ ]:
%%sql
FROM fake_data

# Time travel

In [ ]:
%%sql
FROM fake_data

In [ ]:
%%sql
SELECT current_catalog()

In [ ]:
%%sql
USE '__ducklake_metadata_my_ducklake'

In [ ]:
%%sql
FROM ducklake_snapshot_changes

In [ ]:
%%sql
FROM ducklake_snapshot

In [ ]:
%%sql
USE 'my_ducklake';

In [ ]:
%%sql
SELECT * FROM fake_data AT (VERSION => 1);